## Section 1: SparkSession Setup

In [1]:
import findspark
findspark.init('/opt/spark')

In [2]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import (StructType, StructField, StringType,
                                IntegerType, DoubleType, DateType)

spark = SparkSession.builder \
    .appName('Week6') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()

print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print('SparkSession ready!')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/05 15:48:09 WARN Utils: Your hostname, Desktop, resolves to a loopback address: 127.0.1.1; using 10.131.65.154 instead (on interface wlo1)
26/07/05 15:48:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/05 15:48:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2
App name: Week6
Master: local[*]
SparkSession ready!


In [3]:
spark

---
## Section 2: Lazy Evaluation & DAG (Lineage Graph)


In [4]:
df_lazy = spark.read.csv('retail_customers.csv', header=True, inferSchema=True)
df_filtered = df_lazy.filter(f.col('Age') > 25)
df_selected = df_filtered.select('CustomerID', 'Age', 'Country')
df_renamed = df_selected.withColumnRenamed('CustomerID', 'CustID')

print('No action called yet - nothing has been computed!')
print(f'Type: {type(df_renamed)}')

print(f'\nRow count (ACTION triggers execution): {df_renamed.count()}')
df_renamed.show(5)

No action called yet - nothing has been computed!
Type: <class 'pyspark.sql.classic.dataframe.DataFrame'>

Row count (ACTION triggers execution): 6208
+---------+---+-------+
|   CustID|Age|Country|
+---------+---+-------+
|CUST00001| 33|  India|
|CUST00002| 40|    USA|
|CUST00003| 60|     UK|
|CUST00006| 33|  India|
|CUST00007| 39|Germany|
+---------+---+-------+
only showing top 5 rows


In [5]:
# View  plan
print('=== Spark plan ===')
df_renamed.explain(True)

=== Spark plan ===
== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumnsrenames(CustomerID, CustID)]
+- Project [CustomerID#17, Age#19, Country#22]
   +- Filter (Age#19 > 25)
      +- Relation [CustomerID#17,RegistrationDate#18,Age#19,Gender#20,IncomeLevel#21,Country#22,City#23,TotalPurchases#24,AvgOrderValue#25,CustomerLifetimeValue#26,FavoriteCategory#27,SatisfactionScore#28,RepeatCustomer#29,PremiumMember#30,HasReturned#31,LoyaltyScore#32,LastPurchaseDate#33] csv

== Analyzed Logical Plan ==
CustID: string, Age: int, Country: string
Project [CustomerID#17 AS CustID#35, Age#19, Country#22]
+- Project [CustomerID#17, Age#19, Country#22]
   +- Filter (Age#19 > 25)
      +- Relation [CustomerID#17,RegistrationDate#18,Age#19,Gender#20,IncomeLevel#21,Country#22,City#23,TotalPurchases#24,AvgOrderValue#25,CustomerLifetimeValue#26,FavoriteCategory#27,SatisfactionScore#28,RepeatCustomer#29,PremiumMember#30,HasReturned#31,LoyaltyScore#32,LastPurchaseDate#33] csv

== Optimized Logical P

---
## Section 3: Reading Data with Proper Schema Handling

In [6]:
sschema = StructType([
    StructField('CustomerID', StringType(), True),
    StructField('RegistrationDate', StringType(), True),
    StructField('Age', IntegerType(), True),
    StructField('Gender', StringType(), True),
    StructField('IncomeLevel', StringType(), True),
    StructField('Country', StringType(), True),
    StructField('City', StringType(), True),
    StructField('TotalPurchases', IntegerType(), True),
    StructField('AvgOrderValue', DoubleType(), True),
    StructField('CustomerLifetimeValue', DoubleType(), True),
    StructField('FavoriteCategory', StringType(), True),
    StructField('SatisfactionScore', DoubleType(), True),
    StructField('RepeatCustomer', StringType(), True),
    StructField('PremiumMember', StringType(), True),
    StructField('HasReturned', StringType(), True),
    StructField('LoyaltyScore', DoubleType(), True),
    StructField('LastPurchaseDate', StringType(), True),
])
dfs = spark.read.csv('retail_customers.csv', header=True, schema=sschema)
dfs.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- RegistrationDate: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- IncomeLevel: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- TotalPurchases: integer (nullable = true)
 |-- AvgOrderValue: double (nullable = true)
 |-- CustomerLifetimeValue: double (nullable = true)
 |-- FavoriteCategory: string (nullable = true)
 |-- SatisfactionScore: double (nullable = true)
 |-- RepeatCustomer: string (nullable = true)
 |-- PremiumMember: string (nullable = true)
 |-- HasReturned: string (nullable = true)
 |-- LoyaltyScore: double (nullable = true)
 |-- LastPurchaseDate: string (nullable = true)



---
## Section 4: Filtering & Column Selection

In [7]:
# Load the main working DataFrame
raw_df = spark.read.csv('retail_customers.csv', header=True, schema=sschema)
print(f'Loaded: {raw_df.count()} rows, {len(raw_df.columns)} columns')
raw_df.show(5, truncate=False)

Loaded: 8000 rows, 17 columns
+----------+----------------+----+------+-----------+---------+-------+--------------+-------------+---------------------+----------------+-----------------+--------------+-------------+-----------+------------+----------------+
|CustomerID|RegistrationDate|Age |Gender|IncomeLevel|Country  |City   |TotalPurchases|AvgOrderValue|CustomerLifetimeValue|FavoriteCategory|SatisfactionScore|RepeatCustomer|PremiumMember|HasReturned|LoyaltyScore|LastPurchaseDate|
+----------+----------------+----+------+-----------+---------+-------+--------------+-------------+---------------------+----------------+-----------------+--------------+-------------+-----------+------------+----------------+
|CUST00001 |2020-02-21      |33  |Female|Medium     |India    |Chicago|7             |23.27        |218.58               |Toys            |6.3              |Yes           |Yes          |No         |15.55       |2024-06-21      |
|CUST00002 |2022-02-17      |40  |NULL  |High       |U

In [8]:
selected = raw_df.select('CustomerID', 'Age', 'Country', 'AvgOrderValue', 'SatisfactionScore')
selected.show(5)

adults = raw_df.filter((f.col('Age') >= 18) & (f.col('Age') <= 65))
print(f'Adults (18-65): {adults.count()}')

electronics = raw_df.filter(f.col('FavoriteCategory') == 'Electronics')
print(f'Electronics fans: {electronics.count()}')

high_value = raw_df.filter(
    (f.col('PremiumMember') == 'Yes') &
    (f.col('IncomeLevel').isin('High', 'Very High')) &
    (f.col('SatisfactionScore') > 7.0)
)
print(f'Premium high-value customers: {high_value.count()}')
high_value.select('CustomerID','Country','IncomeLevel','SatisfactionScore').show(5)

+----------+----+---------+-------------+-----------------+
|CustomerID| Age|  Country|AvgOrderValue|SatisfactionScore|
+----------+----+---------+-------------+-----------------+
| CUST00001|  33|    India|        23.27|              6.3|
| CUST00002|  40|      USA|       112.65|             NULL|
| CUST00003|  60|       UK|       185.45|             1.29|
| CUST00004|NULL|    Japan|       199.48|             8.13|
| CUST00005|  25|Australia|       438.55|             5.57|
+----------+----+---------+-------------+-----------------+
only showing top 5 rows
Adults (18-65): 7005
Electronics fans: 945
Premium high-value customers: 525
+----------+---------+-----------+-----------------+
|CustomerID|  Country|IncomeLevel|SatisfactionScore|
+----------+---------+-----------+-----------------+
| CUST00038|    India|  Very High|              8.5|
| CUST00111|Australia|  Very High|             9.92|
|      NULL|       UK|  Very High|             9.67|
| CUST00140|       UK|       High|       

---
## Section 5: DataFrame Modifications (Rename, Cast, Add Columns)

In [9]:
df_mod = raw_df \
    .withColumnRenamed('CustomerLifetimeValue', 'CLV') \
    .withColumnRenamed('SatisfactionScore', 'Satisfaction') \
    .withColumnRenamed('AvgOrderValue', 'AOV')

df_mod = df_mod \
    .withColumn('RegistrationDate', f.to_date('RegistrationDate', 'yyyy-MM-dd')) \
    .withColumn('LastPurchaseDate', f.to_date('LastPurchaseDate', 'yyyy-MM-dd'))

df_mod = df_mod \
    .withColumn('EstRevenue', f.round(f.col('AOV') * f.col('TotalPurchases'), 2)) \
    .withColumn('DaysSinceRegistration',
               f.datediff(f.current_date(), f.col('RegistrationDate'))) \
    .withColumn('ValueSegment',
               f.when(f.col('CLV') > 1000, 'High')
                .when(f.col('CLV') > 300, 'Medium')
                .otherwise('Low'))

df_mod.select('CustomerID','AOV','TotalPurchases','EstRevenue',
              'DaysSinceRegistration','ValueSegment','CLV').show(10)
df_mod.printSchema()

+----------+------+--------------+----------+---------------------+------------+-------+
|CustomerID|   AOV|TotalPurchases|EstRevenue|DaysSinceRegistration|ValueSegment|    CLV|
+----------+------+--------------+----------+---------------------+------------+-------+
| CUST00001| 23.27|             7|    162.89|                 2326|         Low| 218.58|
| CUST00002|112.65|            15|   1689.75|                 1599|        High|3028.98|
| CUST00003|185.45|             7|   1298.15|                 1619|        High|1280.53|
| CUST00004|199.48|             7|   1396.36|                 1567|        High| 2281.2|
| CUST00005|438.55|             1|    438.55|                 2354|      Medium|  336.8|
| CUST00006| 30.04|             9|    270.36|                 1748|         Low| 242.67|
| CUST00007| 18.14|            11|    199.54|                 1870|         Low| 253.66|
| CUST00008| 42.09|             2|     84.18|                 1536|         Low|  77.26|
| CUST00009| 62.86|  

---
## Section 6: Null Value Handling

In [11]:
print('=== Null counts per column ===')
null_counts = df_mod.select(
    [f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_mod.columns]
)
null_counts.show(truncate=False)

before = df_mod.count()
df_clean = df_mod.filter(f.col('CustomerID').isNotNull())
after = df_clean.count()
print(f'Dropped {before - after} rows with null CustomerID. Remaining: {after}')

df_clean = df_clean.withColumn('Age',
    f.when((f.col('Age') < 1) | (f.col('Age') > 100), None).otherwise(f.col('Age')))

df_clean = df_clean.withColumn('AOV',
    f.when(f.col('AOV') > 5000, None).otherwise(f.col('AOV')))

num_cols = ['Age','TotalPurchases','AOV','CLV','Satisfaction','LoyaltyScore']
means = {}
for c in num_cols:
    v = df_clean.select(f.avg(c)).first()[0]
    if v: means[c] = round(v, 2)
df_clean = df_clean.fillna(means)
print(f'Imputed numeric columns with means: {means}')

df_clean = df_clean.fillna('Unknown', subset=['Gender','IncomeLevel','FavoriteCategory','City'])
df_clean = df_clean.fillna('No', subset=['RepeatCustomer'])

print('\n=== Null counts after cleaning ===')
df_clean.select(
    [f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_clean.columns]
).show(truncate=False)

=== Null counts per column ===
+----------+----------------+---+------+-----------+-------+----+--------------+---+---+----------------+------------+--------------+-------------+-----------+------------+----------------+----------+---------------------+------------+
|CustomerID|RegistrationDate|Age|Gender|IncomeLevel|Country|City|TotalPurchases|AOV|CLV|FavoriteCategory|Satisfaction|RepeatCustomer|PremiumMember|HasReturned|LoyaltyScore|LastPurchaseDate|EstRevenue|DaysSinceRegistration|ValueSegment|
+----------+----------------+---+------+-----------+-------+----+--------------+---+---+----------------+------------+--------------+-------------+-----------+------------+----------------+----------+---------------------+------------+
|240       |0               |314|1539  |1594       |0      |440 |372           |0  |329|481             |417         |250           |0            |0          |313         |479             |372       |0                    |0           |
+----------+-------------

---
## Section 7: Transformations vs Actions


In [12]:
narrow_result = df_clean \
    .select('CustomerID', 'Country', 'AOV', 'CLV') \
    .filter(f.col('AOV') > 100) \
    .withColumn('AOV_Doubled', f.col('AOV') * 2)

wide_result = df_clean.groupBy('Country').agg(
    f.count('CustomerID').alias('customers'),
    f.round(f.avg('AOV'), 2).alias('avg_order'),
    f.round(f.sum('EstRevenue'), 2).alias('total_revenue')
).orderBy(f.desc('total_revenue'))

wide_result.show()

+---------+---------+---------+-------------+
|  Country|customers|avg_order|total_revenue|
+---------+---------+---------+-------------+
|       UK|     1039|   115.87|   7732047.64|
|      USA|      926|   112.28|   7010511.39|
|    India|      945|   115.88|   6264152.37|
|    Japan|      967|   115.41|   5808661.84|
|   France|      982|   115.08|   5287993.46|
|Australia|      968|    117.2|    4721695.2|
|  Germany|      983|   108.38|   4641847.54|
|   Canada|      950|   113.55|   3514349.83|
+---------+---------+---------+-------------+



---
## Section 8: Shuffle, Partitioning & Predicate Pushdown

In [13]:
print('=== Physical Plan (notice Exchange = shuffle) ===')
df_clean.groupBy('Country').agg(f.avg('CLV')).explain()

=== Physical Plan (notice Exchange = shuffle) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Country#79], functions=[avg(CLV#644)])
   +- Exchange hashpartitioning(Country#79, 8), ENSURE_REQUIREMENTS, [plan_id=852]
      +- HashAggregate(keys=[Country#79], functions=[partial_avg(CLV#644)])
         +- Project [Country#79, coalesce(nanvl(CustomerLifetimeValue#83, null), 7278.16) AS CLV#644]
            +- Filter isnotnull(CustomerID#74)
               +- FileScan csv [CustomerID#74,Country#79,CustomerLifetimeValue#83] Batched: false, DataFilters: [isnotnull(CustomerID#74)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/nilesh-sharma/dev/Celebal_I/Week_6/retail_customers.csv], PartitionFilters: [], PushedFilters: [IsNotNull(CustomerID)], ReadSchema: struct<CustomerID:string,Country:string,CustomerLifetimeValue:double>




In [14]:
print(f'Default partitions: {df_clean.rdd.getNumPartitions()}')

df_repart = df_clean.repartition(4, 'Country')
print(f'After repartition(4): {df_repart.rdd.getNumPartitions()}')

df_coal = df_repart.coalesce(2)
print(f'After coalesce(2): {df_coal.rdd.getNumPartitions()}')

Default partitions: 1
After repartition(4): 4
After coalesce(2): 2


In [15]:
df_clean.write.mode('overwrite').parquet('output/temp_parquet')

df_pushed = spark.read.parquet('output/temp_parquet') \
    .filter(f.col('Country') == 'USA')

print('=== Plan showing Predicate Pushdown ===')
print('(Look for PushedFilters in the physical plan)')
df_pushed.explain()

=== Plan showing Predicate Pushdown ===
(Look for PushedFilters in the physical plan)
== Physical Plan ==
*(1) Filter (isnotnull(Country#961) AND (Country#961 = USA))
+- *(1) ColumnarToRow
   +- FileScan parquet [CustomerID#956,RegistrationDate#957,Age#958,Gender#959,IncomeLevel#960,Country#961,City#962,TotalPurchases#963,AOV#964,CLV#965,FavoriteCategory#966,Satisfaction#967,RepeatCustomer#968,PremiumMember#969,HasReturned#970,LoyaltyScore#971,LastPurchaseDate#972,EstRevenue#973,DaysSinceRegistration#974,ValueSegment#975] Batched: true, DataFilters: [isnotnull(Country#961), (Country#961 = USA)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/nilesh-sharma/dev/Celebal_I/Week_6/output/temp_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(Country), EqualTo(Country,USA)], ReadSchema: struct<CustomerID:string,RegistrationDate:date,Age:int,Gender:string,IncomeLevel:string,Country:s...




---
## Section 9: CSV vs Parquet – Performance Comparison


In [16]:
# Write as both formats
df_clean.write.mode('overwrite').option('header', True).csv('output/data_csv')
df_clean.write.mode('overwrite').parquet('output/data_parquet')

# Compare file sizes
import subprocess
csv_size = subprocess.check_output('du -sh output/data_csv', shell=True).decode().split()[0]
parquet_size = subprocess.check_output('du -sh output/data_parquet', shell=True).decode().split()[0]
print(f'CSV size:     {csv_size}')
print(f'Parquet size: {parquet_size}')

CSV size:     964K
Parquet size: 292K


In [ ]:
# Read performance comparison
# CSV read
t1 = time.time()
csv_df = spark.read.csv('output/data_csv', header=True, inferSchema=True)
csv_df.filter(f.col('Country') == 'India').select('CustomerID','AOV').count()
csv_time = time.time() - t1

# Parquet read (with column pruning + predicate pushdown)
t2 = time.time()
pq_df = spark.read.parquet('output/data_parquet')
pq_df.filter(f.col('Country') == 'India').select('CustomerID','AOV').count()
pq_time = time.time() - t2

print(f'CSV read+filter:     {csv_time:.3f}s')
print(f'Parquet read+filter: {pq_time:.3f}s')
print(f'Parquet is {csv_time/pq_time:.1f}x faster')

print('\nParquet schema (auto-detected, no inferSchema needed):')
pq_df.printSchema()

CSV read+filter:     0.588s
Parquet read+filter: 0.402s
Parquet is 1.5x faster

Parquet schema (auto-detected, no inferSchema needed):
root
 |-- CustomerID: string (nullable = true)
 |-- RegistrationDate: date (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- IncomeLevel: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- TotalPurchases: integer (nullable = true)
 |-- AOV: double (nullable = true)
 |-- CLV: double (nullable = true)
 |-- FavoriteCategory: string (nullable = true)
 |-- Satisfaction: double (nullable = true)
 |-- RepeatCustomer: string (nullable = true)
 |-- PremiumMember: string (nullable = true)
 |-- HasReturned: string (nullable = true)
 |-- LoyaltyScore: double (nullable = true)
 |-- LastPurchaseDate: date (nullable = true)
 |-- EstRevenue: double (nullable = true)
 |-- DaysSinceRegistration: integer (nullable = true)
 |-- ValueSegment: string (nullable = true)



---
## Section 10: Aggregations & GroupBy

In [18]:
df_clean.agg(
    f.count('CustomerID').alias('total_customers'),
    f.sum('TotalPurchases').alias('total_purchases'),
    f.round(f.avg('AOV'), 2).alias('avg_order_value'),
    f.round(f.min('CLV'), 2).alias('min_clv'),
    f.round(f.max('CLV'), 2).alias('max_clv')
).show()

df_clean.groupBy('Country').agg(
    f.count('CustomerID').alias('customers'),
    f.round(f.avg('Satisfaction'), 2).alias('avg_satisfaction'),
    f.round(f.sum('EstRevenue'), 2).alias('total_revenue')
).filter(f.col('avg_satisfaction') > 5.0) \
 .orderBy(f.desc('total_revenue')).show()

df_clean.groupBy('FavoriteCategory').agg(
    f.count('CustomerID').alias('count'),
    f.round(f.avg('AOV'), 2).alias('avg_order'),
    f.round(f.avg('CLV'), 2).alias('avg_clv')
).orderBy(f.desc('avg_clv')).show()

+---------------+---------------+---------------+-------+----------+
|total_customers|total_purchases|avg_order_value|min_clv|   max_clv|
+---------------+---------------+---------------+-------+----------+
|           7760|          62606|         114.22|   4.34|1360278.32|
+---------------+---------------+---------------+-------+----------+

+---------+---------+----------------+-------------+
|  Country|customers|avg_satisfaction|total_revenue|
+---------+---------+----------------+-------------+
|       UK|     1039|            5.55|   7732047.64|
|      USA|      926|            5.51|   7010511.39|
|    India|      945|            5.62|   6264152.37|
|    Japan|      967|            5.53|   5808661.84|
|   France|      982|            5.55|   5287993.46|
|Australia|      968|            5.66|    4721695.2|
|  Germany|      983|            5.54|   4641847.54|
|   Canada|      950|            5.43|   3514349.83|
+---------+---------+----------------+-------------+

+----------------

---
## Section 11: Complete Data Pipeline (Read → Transform → Filter → Write)

In [20]:
def run_pipeline(input_path, output_dir):
    print('[1/5] Reading raw CSV with explicit schema...')
    df = spark.read.csv(input_path, header=True, schema=sschema)
    print(f'      Loaded {df.count()} rows')

    print('[2/5] Cleaning: nulls, outliers, imputation...')
    df = df.filter(f.col('CustomerID').isNotNull())
    df = df.withColumn('Age',
        f.when((f.col('Age') < 1) | (f.col('Age') > 100), None).otherwise(f.col('Age')))
    df = df.withColumn('AvgOrderValue',
        f.when(f.col('AvgOrderValue') > 5000, None).otherwise(f.col('AvgOrderValue')))
    num_imp = {}
    for c in ['Age','TotalPurchases','AvgOrderValue','CustomerLifetimeValue','SatisfactionScore','LoyaltyScore']:
        v = df.select(f.avg(c)).first()[0]
        if v: num_imp[c] = round(v, 2)
    df = df.fillna(num_imp)
    df = df.fillna('Unknown', subset=['Gender','IncomeLevel','FavoriteCategory','City'])
    df = df.fillna('No', subset=['RepeatCustomer'])

    print('[3/5] Transforming: cast, rename, new columns...')
    df = df \
        .withColumn('RegistrationDate', f.to_date('RegistrationDate', 'yyyy-MM-dd')) \
        .withColumn('LastPurchaseDate', f.to_date('LastPurchaseDate', 'yyyy-MM-dd')) \
        .withColumn('EstRevenue', f.round(f.col('AvgOrderValue') * f.col('TotalPurchases'), 2)) \
        .withColumn('ValueSegment',
                    f.when(f.col('CustomerLifetimeValue') > 1000, 'High')
                     .when(f.col('CustomerLifetimeValue') > 300, 'Medium')
                     .otherwise('Low'))

    print('[4/5] Filtering: keeping ages 18-65...')
    df_final = df.filter((f.col('Age') >= 18) & (f.col('Age') <= 65))
    print(f'      Rows after filter: {df_final.count()}')

    print('[5/5] Writing outputs...')

    df_final.write.mode('overwrite').parquet(os.path.join(output_dir, 'cleaned_parquet'))

    cat_rpt = df_final.groupBy('FavoriteCategory').agg(
        f.count('CustomerID').alias('customers'),
        f.round(f.avg('AvgOrderValue'), 2).alias('avg_order'),
        f.round(f.sum('EstRevenue'), 2).alias('total_revenue')
    ).orderBy(f.desc('total_revenue'))
    cat_rpt.coalesce(1).write.mode('overwrite').option('header', True) \
        .csv(os.path.join(output_dir, 'category_summary'))

    cty_rpt = df_final.groupBy('Country').agg(
        f.count('CustomerID').alias('customers'),
        f.round(f.avg('CustomerLifetimeValue'), 2).alias('avg_clv'),
        f.round(f.sum('EstRevenue'), 2).alias('total_revenue')
    ).orderBy(f.desc('total_revenue'))
    cty_rpt.coalesce(1).write.mode('overwrite').option('header', True) \
        .csv(os.path.join(output_dir, 'country_summary'))

    print(f'Pipeline complete! Outputs in {output_dir}/')

run_pipeline('retail_customers.csv', 'output')

[1/5] Reading raw CSV with explicit schema...
      Loaded 8000 rows
[2/5] Cleaning: nulls, outliers, imputation...
[3/5] Transforming: cast, rename, new columns...
[4/5] Filtering: keeping ages 18-65...
      Rows after filter: 7727
[5/5] Writing outputs...
Pipeline complete! Outputs in output/


---
## Section 13: Best Practices for Large Datasets


In [21]:
# Best practice: show() vs collect()
df_clean.select('CustomerID', 'Country', 'AOV').show(5)

# BAD for large data (commented out):
# all_rows = df_clean.collect()  # Pulls ALL rows to driver memory!

+----------+---------+------+
|CustomerID|  Country|   AOV|
+----------+---------+------+
| CUST00001|    India| 23.27|
| CUST00002|      USA|112.65|
| CUST00003|       UK|185.45|
| CUST00004|    Japan|199.48|
| CUST00005|Australia|438.55|
+----------+---------+------+
only showing top 5 rows


---
## Cleanup

In [23]:
spark.stop()